In [34]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_curve
)
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler 
from sklearn.ensemble import RandomForestClassifier


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [48]:
# Load original and new data
df_original = pd.read_csv(r'D:\Complete_proj\Email_spam\model\data\email.csv')
df_new_ham = pd.read_csv(r'D:\Complete_proj\Email_spam\model\data\new_ham_data.csv')

# Combine and shuffle
df_combined = pd.concat([df_original, df_new_ham], ignore_index=True)
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

# Save back to csv
df_combined.to_csv('email_updated.csv', index=False)

In [49]:
data=pd.read_csv(r"D:\Complete_proj\Email_spam\model\data\email_updated.csv")
data.head()

,Category,Message
0,ham,There the size of elephant tablets & u shove u...
1,ham,Customer place i will call you.
2,ham,"Only just got this message, not ignoring you. ..."
3,spam,Your Hulu subscription is about to expire. Ren...
4,ham,Does she usually take fifteen fucking minutes ...


In [50]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6620 entries, 0 to 6619
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  6620 non-null   object
 1   Message   6620 non-null   object
dtypes: object(2)
memory usage: 103.6+ KB


In [51]:
data.Category.value_counts()

Category
ham     4925
spam    1695
Name: count, dtype: int64

In [52]:
duplicated_rows=data[data.duplicated(subset=['Message'], keep=False)]
print(duplicated_rows.sort_values(by='Message').head(10))

     Category                                            Message
5042     spam                                            #ERROR!
3046     spam                                            #ERROR!
1434     spam                                            #ERROR!
3970      ham  1) Go to write msg 2) Put on Dictionary mode 3...
4747      ham  1) Go to write msg 2) Put on Dictionary mode 3...
2075      ham  1) Go to write msg 2) Put on Dictionary mode 3...
2123     spam  18 days to Euro2004 kickoff! U will be kept in...
767      spam  18 days to Euro2004 kickoff! U will be kept in...
2043     spam  4mths half price Orange line rental & latest c...
1921     spam  4mths half price Orange line rental & latest c...


In [53]:
data=data.drop_duplicates(subset=['Message'], keep='first')


In [54]:
data.duplicated(subset=['Message']).sum()

np.int64(0)

In [55]:
data.isnull().sum()

Category    0
Message     0
dtype: int64

In [56]:
print(f"shape of data : {data.shape}")

shape of data : (6142, 2)


In [57]:
mail=data['Message'].astype(str)
target=data['Category']
x_train,x_test,y_train,y_test=train_test_split(mail,target,test_size=0.2,shuffle=True,random_state=42)

mask=y_train.isin(['ham','spam'])
x_train=x_train[mask]
y_train=y_train[mask]

In [58]:
import re
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

def clean_text(text_series):
    # Function to apply cleaning to each string in the series
    def transform(text):
        text = text.lower()
        # Replace URLs
        text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
        # Replace Numbers
        text = re.sub(r'\d+', '[NUM]', text)
        return text
    
    return text_series.apply(transform)

# Wrap it for the pipeline
text_cleaner = FunctionTransformer(clean_text)

In [59]:
pipeline = Pipeline([
    ('cleaner', FunctionTransformer(clean_text)),
    ('tfidf', TfidfVectorizer(
        stop_words=None, 
        ngram_range=(1,2),
        max_features=5000,

    )),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=2,
        max_depth=None,
        class_weight='balanced_subsample'
    ))
])

In [60]:
pipeline.fit(x_train, y_train)

,steps,"[('cleaner', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function cle...001D7D48A0360>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [62]:
model_pred_test=pipeline.predict(x_test)
print(confusion_matrix(y_test, model_pred_test))
print(classification_report(y_test, model_pred_test))

[[932   7]
 [  8 282]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       939
        spam       0.98      0.97      0.97       290

    accuracy                           0.99      1229
   macro avg       0.98      0.98      0.98      1229
weighted avg       0.99      0.99      0.99      1229



In [63]:
filename = 'spam_model_rf2.joblib'
joblib.dump(pipeline, filename)

['spam_model_rf2.joblib']

In [5]:
data=pd.read_csv(r"D:\KaggleCompetitions\Datasets\ai_jobs_market_2025_2026.csv")

In [6]:
data.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,...,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,...,12.0,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,9.4,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


In [7]:
data.shape

(1500, 25)

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_id                 1500 non-null   object 
 1   job_title              1500 non-null   object 
 2   job_category           1500 non-null   object 
 3   experience_level       1500 non-null   object 
 4   years_of_experience    1500 non-null   int64  
 5   education_required     1500 non-null   object 
 6   annual_salary_usd      1500 non-null   float64
 7   salary_min_usd         1500 non-null   int64  
 8   salary_max_usd         1500 non-null   int64  
 9   city                   1500 non-null   object 
 10  country                1500 non-null   object 
 11  remote_work            1500 non-null   object 
 12  company_size           1500 non-null   object 
 13  industry               1500 non-null   object 
 14  required_skills        1500 non-null   object 
 15  ai_s

In [9]:
dataa=data.copy()

In [10]:
dataa['job_category'].value_counts()

job_category
AI Engineering      736
Data Science        127
Governance          122
Robotics             74
Product              70
Business             62
Infrastructure       55
Architecture         52
ML Operations        51
Data Engineering     51
Security             50
Research             50
Name: count, dtype: int64

In [11]:
df=dataa[dataa['job_category'] == 'AI Engineering']['years_of_experience'].mean()
print(f"Average years : {df}")

Average years : 5.5896739130434785


In [12]:
title=(dataa.groupby('job_category')['job_title'].value_counts())

In [13]:
title

job_category      job_title               
AI Engineering    LLM Engineer                75
                  Generative AI Engineer      71
                  Prompt Engineer             71
                  Multimodal AI Engineer      67
                  AI Engineer                 64
                  Senior ML Engineer          64
                  Deep Learning Engineer      58
                  AI Agent Developer          57
                  ML Engineer                 55
                  NLP Engineer                55
                  RAG Engineer                53
                  Computer Vision Engineer    46
Architecture      AI Solutions Architect      52
Business          AI Business Analyst         62
Data Engineering  Data Engineer (AI)          51
Data Science      Senior Data Scientist       66
                  Data Scientist              61
Governance        AI Compliance Manager       66
                  AI Ethics Officer           56
Infrastructure    AI Infra

In [14]:
df = dataa[(dataa['job_category'] == 'AI Engineering') & 
           (dataa['job_title']=='RAG Engineer')]['required_skills'].unique()

print(df)

['Embeddings|LLMs|Vector DBs|Search Systems|LangChain|Python|Communication|Agile|Research'
 'LLMs|Python|Embeddings|Vector DBs|Search Systems|LangChain|Statistics|Agile'
 'Vector DBs|Embeddings|Search Systems|Git'
 'Search Systems|Vector DBs|Python|Embeddings|LangChain|Linux'
 'Search Systems|Vector DBs|Embeddings|LangChain|Linux'
 'Vector DBs|Embeddings|LLMs|Problem Solving|Research|Statistics'
 'LangChain|Python|Embeddings|Vector DBs|LLMs|Statistics|Leadership|SQL'
 'Search Systems|Embeddings|Python|LangChain|Vector DBs|LLMs|Git|SQL'
 'Vector DBs|LangChain|Search Systems|LLMs|Cloud'
 'Search Systems|Python|LLMs|Embeddings|Vector DBs|Problem Solving|Cloud'
 'Embeddings|Search Systems|LangChain|Agile|Linux|Git'
 'Vector DBs|Python|LangChain|Search Systems|Communication'
 'Embeddings|Search Systems|Python|LLMs|LangChain|Vector DBs|Research|Git|Statistics'
 'LangChain|Vector DBs|Python|LLMs|Embeddings|Search Systems|Communication|Statistics|Research'
 'LLMs|Search Systems|Python|Communic

In [15]:
spam_data=pd.read_csv(r"D:\Complete_proj\Email_spam\model\data\enron_spam_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Complete_proj\\Email_spam\\model\\data\\enron_spam_data.csv'

In [ ]:
spam_data.shape

(33716, 5)

In [ ]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  33716 non-null  int64 
 1   Subject     33716 non-null  object
 2   Message     33664 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [ ]:
spam_data

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-29
33712,33712,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-29
33713,33713,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-30
33714,33714,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-30
